# 유튜브 채널 수집기 — 메타데이터 · 자막

「안녕하세요원이입니다잘부탁드립니다」(`@helloiamwoninicetomeetyou`) 채널의
**영상 메타데이터 · 자동자막**을 받아 분석용 표로 정리합니다.
채널 핸들이나 ID만 바꾸면 다른 채널에도 그대로 쓸 수 있습니다.

---

## 산출 폴더 구조

```
{OUT_ROOT}/
├ 01_RAW/{MEDIA_DIR}/                 원자료 (원형 보존, 이후 절대 수정하지 않음)
│  ├ {stem}.info.json                     yt-dlp 전체 메타데이터
│  ├ {stem}.ko-orig.json3                 자동자막 원본 (밀리초 타임스탬프)
│  └ raw_xlsx/                            영상단위·챕터단위 날것 표
├ 02_METADATA/                        보정본 반영 메타데이터 (xlsx + csv 정본)
├ 03_TEXT/{MEDIA_DIR}/                분석용 자막 txt (원본 2판본 + 보정 2판본)
└ 04_LOG/                             수집 로그 · 치환 로그 · 보정 사전 · 캐시
```

## 실행 순서

| 단계 | 내용 | 네트워크 | 오래 걸림 |
|---|---|---|---|
| 0 | 환경 준비 | ○ | |
| 1 | 설정 · 프리셋 | | |
| 2 | 공통 유틸 | | |
| 3 | 채널 확인 · 폴더 준비 | ○ | |
| 4 | 영상 열거 | ○ | △ |
| 5 | 메타데이터·자막 | ○ | ★ |
| 6 | 파생 프로그램 (기본 꺼짐) | ○ | |
| 7 | ASR 고유명사 보정 | | |
| 8~9 | 표 생성 + txt 경로 부착 | | |
| 10 | 검증 | | |

4~6단계는 중단되어도 **다시 실행하면 이미 받은 것은 건너뜁니다.** 마음 편히 재실행하세요.


## 0. 환경 준비

`yt-dlp`는 유튜브 변경에 맞춰 자주 갱신됩니다. **수집 전에 반드시 최신으로 올려주세요.**
버전이 낮으면 자막이 빈 채로 내려오거나 채널 목록 조회가 통째로 실패합니다.

In [ ]:
%pip install -q --upgrade yt-dlp pandas openpyxl

import subprocess, shutil, sys
_exe = shutil.which("yt-dlp")
_cmd = [_exe, "--version"] if _exe else [sys.executable, "-m", "yt_dlp", "--version"]
print("yt-dlp", subprocess.run(_cmd, capture_output=True, text=True).stdout.strip())

## 1. 설정

### 채널 지정

`CHANNEL`에는 아래 중 **아무거나** 넣으면 됩니다. 3단계에서 채널 ID로 해석해 확인시켜 드립니다.

| 형태 | 예 |
|---|---|
| 핸들 | `@helloiamwoninicetomeetyou` |
| 핸들 (@ 없이) | `helloiamwoninicetomeetyou` |
| 채널 ID | `UCWpY0eSJtyO-qNAPbKFRSSg` |
| 채널 URL | `https://www.youtube.com/@helloiamwoninicetomeetyou` |
| 탭이 붙은 URL | `.../@helloiamwoninicetomeetyou/videos` |
| 구형 URL | `/c/이름` · `/user/이름` |

**핸들 찾는 법** — 채널 페이지 주소창의 `youtube.com/` 뒤 `@`로 시작하는 부분입니다.

> 핸들은 채널 주인이 바꿀 수 있지만 **채널 ID(`UC`+22자)는 바뀌지 않습니다.**
> 이 노트북은 입력이 무엇이든 채널 ID로 바꿔 요청하고, 3단계에서 그 ID를 찍어줍니다.
> 나중에 수집을 재현하실 때를 위해 그 ID를 기록해 두세요.

### 수집 대상 고르기 — `PERIOD_MODE`

| 값 | 동작 |
|---|---|
| `"date"` | `START_DATE ~ END_DATE` 사이 영상 (기본) |
| `"latest"` | 날짜 상관없이 **최신 `MAX_VIDEOS`편** |

### 날짜를 어디서 읽을지 — `DATE_SOURCE`

| 값 | 동작 |
|---|---|
| `"title"` | 제목의 `YYYY년 M월 D일`에서 방송일을 뽑습니다. **목록 조회만으로 끝나 빠릅니다** |
| `"upload_date"` | 유튜브 업로드 날짜. 제목에 날짜가 없는 일반 채널은 이쪽입니다 |

`"upload_date"`는 목록 조회로 날짜를 알 수 없어 **영상마다 한 번씩 더 조회**합니다(편당 1~2초).
대신 최신순으로 훑다가 기간보다 오래된 영상이 `PROBE_STOP_AFTER`편 연속 나오면 멈추고,
조회한 날짜는 `04_LOG/upload_date_cache.json`에 캐시해 재실행 시 다시 묻지 않습니다.


In [ ]:
import os, re

# 프리셋: 채널마다 다른 부분(본방송 판별·파생 프로그램·ASR 사전)의 묶음
#   "generic" = 아무 채널. 모든 영상을 동등하게 취급
PRESET = "generic"

CONFIG = {

    # ═══ 출력 위치 ═══════════════════════════════════════════════════════════
    # 결과가 저장될 폴더. 없으면 새로 만듭니다.
    # 채널마다 다른 폴더로 두세요 — 같은 폴더에 두 채널을 받으면 섞입니다.
    # os.getcwd() 는 "이 노트북이 있는 폴더". 절대경로를 직접 적어도 됩니다.
    "OUT_ROOT":   os.path.join(os.getcwd(), "youtube_woni"),

    # ═══ 채널 ════════════════════════════════════════════════════════════════
    # 핸들(@...) · 채널ID(UC...) · 채널 URL 아무거나. 3단계에서 채널ID로 해석합니다.
    "CHANNEL":    "@helloiamwoninicetomeetyou",

    # 훑을 채널 탭. 이 채널은 /streams(라이브) 탭이 없어서 videos 만 넣었습니다.
    #   "videos"  = 일반 영상 (이 채널 30편)
    #   "streams" = 라이브 다시보기  ← 이 채널에는 없음. 넣으면 "조회 실패" 메시지
    #   "shorts"  = 쇼츠 (이 채널 78편). 넣으려면 MAX_VIDEOS 도 함께 올리세요
    # ※ 원소가 하나일 때 뒤의 쉼표를 빠뜨리면 안 됩니다. ("videos") 는 문자열입니다
    "TABS":       ("videos",),

    # ═══ 수집 범위 ═══════════════════════════════════════════════════════════
    # "date"   = START_DATE ~ END_DATE 사이 영상만
    # "latest" = 날짜 상관없이 최신 MAX_VIDEOS 편
    # 이 채널은 제목에 날짜가 전혀 없고 영상도 30편뿐이라 latest 로 전수 수집합니다
    "PERIOD_MODE": "latest",

    # PERIOD_MODE="date" 일 때만 쓰입니다. 지금은 무시되는 값입니다
    # (다만 04_LOG 의 수집 기록에는 그대로 적히니, 헷갈리지 않게 지워두셔도 됩니다)
    "START_DATE":  "20260723",
    "END_DATE":    "20260807",

    # PERIOD_MODE="latest" 일 때 최신 몇 편을 받을지.
    # 실제 30편이라 100 이면 전부 받습니다. 쇼츠까지 넣으면 200 으로 올리세요
    "MAX_VIDEOS":  5,

    # 채널 목록을 위에서부터 몇 개까지 읽어올지 (탭당).
    # 목록은 최신순이라, 오래된 기간을 수집하려면 이 값이 커야 그 지점까지 내려갑니다.
    # 요청은 한 번뿐이라 크게 둬도 느려지지 않습니다. 이 채널은 30편이라 여유롭습니다
    "ENUM_LIMIT":  400,

    # PERIOD_MODE="date" + DATE_SOURCE="upload_date" 일 때만 작동합니다.
    # 목록을 최신순으로 내려가다 "기간보다 오래된 영상"이 이만큼 연속 나오면 중단합니다.
    # 1 이 아니라 30 인 이유 — 고정 영상·예약 공개 때문에 목록 순서가 흐트러질 수 있어
    # 한 편 봤다고 멈추면 그 뒤의 기간 내 영상을 놓칩니다.  지금은 쓰이지 않는 값입니다
    "PROBE_STOP_AFTER": 30,

    # ═══ 자막 ════════════════════════════════════════════════════════════════
    # ko-orig = 한국어 원어 자동자막(원본).  ko = 번역을 거친 판본
    # 앞에 적은 것이 우선이며, 둘 다 있으면 둘 다 받습니다
    "SUB_LANGS":  "ko-orig,ko",

    # json3 은 밀리초 타임스탬프를 보존합니다. srt/vtt 보다 손실이 적어 이걸 씁니다
    "SUB_FORMAT": "json3",

    # True 로 두면 채널이 직접 올린 수동 자막이 있을 때 그것도 함께 받습니다.
    # 자동자막보다 정확하지만 대부분의 채널에는 없습니다
    "WRITE_MANUAL_SUBS": False,

    # ═══ 요청 ════════════════════════════════════════════════════════════════
    # 요청과 요청 사이 쉬는 시간(초). 짧으면 빠르지만 유튜브가 봇으로 의심합니다.
    # "Sign in to confirm you're not a bot" 이 뜨면 5~10 으로 올리세요
    "REQ_DELAY":  3.0,

    # 영상 메타데이터 조회 / 채널 목록 조회의 제한시간(초)
    "TIMEOUT_INFO": 180,

    # 자막 내려받기의 제한시간(초). 긴 영상은 자막도 커서 따로 둡니다
    "TIMEOUT_SUB":  240,

    # 봇 확인·연령제한에 막힐 때 브라우저의 로그인 쿠키를 빌려옵니다.
    # "chrome" / "edge" / "firefox" 중 평소 유튜브에 로그인해 쓰시는 브라우저 이름.
    # 평소에는 None 으로 두고, 막혔을 때만 켜세요
    "COOKIES_FROM_BROWSER": None,

    # ═══ 표 ══════════════════════════════════════════════════════════════════
    # CSV·txt 인코딩. utf-8-sig 는 BOM 이 붙어 Excel 에서 한글이 안 깨집니다.
    # 읽을 때도 pd.read_csv(..., encoding="utf-8-sig") 로 맞춰주세요
    "ENC": "utf-8-sig",

    # True 면 표 안에 자막 전문(text 열)을 넣습니다.
    # 기본 False — Excel 한 셀 상한이 32,767자라 긴 자막은 행 구조를 깨뜨립니다.
    # 전문은 txt_flat / txt_full / *_corrected 열이 가리키는 파일에 온전히 있습니다
    "TABLE_INCLUDE_TEXT": False,

    # 위를 True 로 켰을 때, 자막을 몇 자씩 잘라 text / text_2 / text_3 에 나눠 담을지.
    # Excel 상한 32,767 보다 조금 작게 둔 값입니다
    "CELL_MAX": 32000,

    # 표가 이 행수를 넘으면 xlsx 저장을 건너뛰고 csv 만 만듭니다 (xlsx 쓰기가 매우 느려지므로)
    "XLSX_ROW_LIMIT": 200000,

    # ═══ 검증용 (10단계) ═════════════════════════════════════════════════════
    # 기존에 같은 방식으로 받아둔 폴더가 있으면 그 경로를 넣으세요.
    # 파일명·열 구성을 대조해 줍니다. 처음 수집이면 "" 로 비워둡니다
    "REFERENCE_ROOT": "",
}

# 프리셋이 비워둔 값(None)은 3단계에서 채널 핸들로 자동 결정됩니다.
# 그대로 두면 HELLOIAMWONINICETOMEETYOU / helloiamwoninicetomeetyou 가 되어
# 폴더명·파일명이 길어지므로, 짧게 쓰고 싶다면 여기서 덮어씁니다.
OVERRIDE = {
    "MEDIA_DIR":  "WONI",      # 01_RAW/WONI/ · 03_TEXT/WONI/ 폴더 이름
    "SLUG":       "woni",      # 파일명 조각: 20260311_woni_GJuMbGXXedY_full.txt
                               #             표 이름: woni_broadcasts.csv
    "MEDIA_NAME": "안원잘부",   # 표의 media 열 값 + RAW_안원잘부_방송단위.csv
}

print(f"PRESET   {PRESET}")
print(f"CHANNEL  {CONFIG['CHANNEL']}")
print(f"범위     {CONFIG['PERIOD_MODE']}"
      + (f"  {CONFIG['START_DATE']}~{CONFIG['END_DATE']}" if CONFIG["PERIOD_MODE"] == "date"
         else f"  최신 {CONFIG['MAX_VIDEOS']}편"))


### 프리셋

채널마다 달라지는 것들 — 본방송을 어떻게 알아보는지, 파생 프로그램이 있는지,
자막의 어떤 고유명사를 고쳐야 하는지 — 을 모아둔 묶음입니다. 지금은 `generic` 하나만 있습니다.

#### `generic`

아무 채널에나 씁니다. **채널 안의 모든 영상을 동등하게 취급**하고,
폴더명·파일명은 3단계에서 채널 핸들을 보고 자동으로 정합니다(위 `OVERRIDE`로 덮어쓸 수 있습니다).
하루 여러 편이 올라올 수 있어 파일명에 video_id를 넣습니다
(예: `20260828_woni_GJuMbGXXedY_full.txt`).

ASR 보정 사전(`CORRECTIONS` · `CONTEXT_CORRECTIONS` · `PRESERVED`)은 비어 있습니다.
자막에서 자주 틀리는 고유명사가 보이면 `(변이형, 치환형, 표준형, confidence, 근거)` 형식으로
`CORRECTIONS`에 추가하세요. 치환은 문자열 교체라 본문을 다시 쓰지 않으며,
변이형끼리 부분 문자열 관계일 수 있어 **긴 것부터** 치환합니다.

> 보정은 **자막에만** 적용되고, `_corrected` 별도 파일로 저장되어 원본과 대조할 수 있습니다.


In [ ]:
PRESETS = {

    "generic": {
        # None 인 값은 3단계에서 채널 정보로 자동 결정
        "MEDIA_NAME": None, "MEDIA_DIR": None, "SLUG": None, "PROGRAM_NAME": None,
        "MAIN_SHOW_PATTERN": None,      # None = 채널의 모든 영상을 동등 취급
        "TITLE_DATE_PATTERN": r"(\d{4})[.\-년]\s*(\d{1,2})[.\-월]\s*(\d{1,2})",
        "DATE_SOURCE": "upload_date",
        "ONE_MAIN_PER_DAY": False,      # 하루 여러 편이 정상
        "STEM_WITH_VIDEO_ID": True,     # 파일명에 video_id 포함 (충돌 방지)
        "SEGMENT_PREFIX": None,         # None = SLUG 앞 3글자를 대문자로
        "COLLECT_SISTER": False,
        "PROG_CODE": {},
        "CORRECTIONS": [], "CONTEXT_CORRECTIONS": [], "PRESERVED": [],
        "RECALL_TERMS": [],
    },
}

if PRESET not in PRESETS:
    raise ValueError(f"PRESET 은 {list(PRESETS)} 중 하나여야 합니다 (받은 값: {PRESET!r})")
CONFIG.update(PRESETS[PRESET])
CONFIG.update(OVERRIDE)

CORRECTIONS         = CONFIG["CORRECTIONS"]
CONTEXT_CORRECTIONS = CONFIG["CONTEXT_CORRECTIONS"]
PRESERVED           = CONFIG["PRESERVED"]
RECALL_TERMS        = CONFIG["RECALL_TERMS"]

print(f"프리셋 '{PRESET}' 적용")
print(f"  본방송 판별   {CONFIG['MAIN_SHOW_PATTERN'] or '(전 영상 동등 취급)'}")
print(f"  파생 프로그램 {'수집' if CONFIG['COLLECT_SISTER'] else '안 함'}")
print(f"  ASR 보정      문자열 {len(CORRECTIONS)}건 · 문맥조건 {len(CONTEXT_CORRECTIONS)}건"
      f" · 보존 {len(PRESERVED)}건")

## 2. 공통 유틸

여기서는 함수만 정의합니다. 폴더는 채널을 확인한 뒤 3단계에서 만듭니다
(`generic` 프리셋은 폴더 이름 자체가 채널에 따라 정해지기 때문입니다).

In [ ]:
import os, re, sys, json, csv, glob, time, shutil, subprocess, datetime
from collections import Counter, defaultdict
import pandas as pd

ENC     = CONFIG["ENC"]
WEEKDAY = "월화수목금토일"
SEP     = "=" * 78
TABS_ALL = ("videos", "streams", "shorts", "playlists", "community", "featured", "about", "live")

MAIN_PAT  = re.compile(CONFIG["MAIN_SHOW_PATTERN"]) if CONFIG["MAIN_SHOW_PATTERN"] else None
TDATE_PAT = re.compile(CONFIG["TITLE_DATE_PATTERN"])


def ytdlp(*args, timeout=180):
    # yt-dlp 실행. 실행파일이 없으면 python -m yt_dlp 로 대체
    exe = shutil.which("yt-dlp")
    cmd = [exe] if exe else [sys.executable, "-m", "yt_dlp"]
    if CONFIG["COOKIES_FROM_BROWSER"]:
        cmd += ["--cookies-from-browser", CONFIG["COOKIES_FROM_BROWSER"]]
    return subprocess.run(cmd + list(args), capture_output=True, text=True,
                          timeout=timeout, encoding="utf-8", errors="replace")


def hms(sec):
    s = int(sec or 0)
    return f"{s//3600:02d}:{s%3600//60:02d}:{s%60:02d}"


def mmss(sec):
    s = int(sec or 0)
    return f"{s//60}분 {s%60}초"


def ymd_dash(ymd):
    return f"{ymd[:4]}-{ymd[4:6]}-{ymd[6:]}" if ymd and len(ymd) == 8 else ""


def wd_of(ymd):
    try:
        return WEEKDAY[datetime.datetime.strptime(ymd, "%Y%m%d").weekday()]
    except Exception:
        return ""


def in_period(ymd):
    return bool(ymd) and CONFIG["START_DATE"] <= ymd <= CONFIG["END_DATE"]


def expected_days():
    d0 = datetime.datetime.strptime(CONFIG["START_DATE"], "%Y%m%d").date()
    d1 = datetime.datetime.strptime(CONFIG["END_DATE"],   "%Y%m%d").date()
    out = []
    while d0 <= d1:
        out.append((d0.strftime("%Y%m%d"), d0.weekday()))
        d0 += datetime.timedelta(days=1)
    return out


def date_from_title(title):
    m = TDATE_PAT.search(str(title or ""))
    if not m:
        return None
    try:
        y, mo, dd = map(int, m.groups()[:3])
        return datetime.date(y, mo, dd).strftime("%Y%m%d")
    except (ValueError, TypeError):
        return None


def is_main_show(title):
    return True if MAIN_PAT is None else bool(MAIN_PAT.search(str(title or "")))


def program_of(title):
    # 제목에서 파생 프로그램명을 뽑는다
    t = str(title or "")
    m = re.match(r"^\[#?([^\]]+)\]", t)
    if m:
        return m.group(1).strip()
    # "YYYY년 M월 D일(요일) 프로그램명(...)" 형태
    m = re.match(r"^\d{4}년\s*\d{1,2}월\s*\d{1,2}일\([월화수목금토일]\)\s*([^\(\[]+)", t)
    if m:
        return m.group(1).strip()
    return "기타"


def code_of(prog):
    if prog in CONFIG["PROG_CODE"]:
        return CONFIG["PROG_CODE"][prog]
    return "ETC" + re.sub(r"[^A-Za-z0-9가-힣]", "", prog)[:6]


def now_str():
    return datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def friendly_error(e):
    # yt-dlp 오류를 사람이 읽을 안내로 바꾼다
    m = str(e)
    if "not a bot" in m or "Sign in to confirm" in m:
        return ("유튜브가 봇 확인을 요구합니다. 요청이 몰렸을 때 자주 납니다.\n"
                "        · 몇 분 뒤 다시 실행하거나\n"
                "        · CONFIG['REQ_DELAY'] 를 3~5초로 늘리거나\n"
                "        · CONFIG['COOKIES_FROM_BROWSER'] = 'chrome' 로 두고 재실행하세요")
    if "Private video" in m or "unavailable" in m.lower():
        return "비공개·삭제된 영상입니다. 건너뜁니다"
    if "age" in m.lower() and "restrict" in m.lower():
        return "연령제한 영상입니다. COOKIES_FROM_BROWSER 를 설정하세요"
    return m[:160]


def order_cols(df, head):
    # head 에 적힌 순서를 앞세우고 나머지는 뒤에 붙인다
    front = [c for c in head if c in df.columns]
    return df[front + [c for c in df.columns if c not in front]]


def safe_filename(s):
    # 윈도우·맥 양쪽에서 안전한 파일명으로 (공백은 _ 로)
    return re.sub(r"\s+", "_", re.sub(r'[\\/:*?"<>|]', "", str(s))).strip("._") or "channel"


def parse_json3(path):
    # json3 → [(초, 텍스트)]. 화자 전환 표식('>>')은 원형 그대로 둔다
    d = json.load(open(path, encoding="utf-8"))
    out = []
    for e in d.get("events", []):
        if not e.get("segs"):
            continue
        t = "".join(s.get("utf8", "") for s in e["segs"]).replace("\n", " ").strip()
        t = re.sub(r"\s+", " ", t)
        if t:
            out.append((e["tStartMs"] / 1000.0, t))
    return out


# ── 채널 해석 ────────────────────────────────────────────────────────────────
def channel_base(spec):
    # 핸들·채널ID·URL 무엇이 들어와도 탭 없는 채널 기본 URL로 정규화
    s = str(spec or "").strip().rstrip("/")
    if not s:
        raise ValueError("CONFIG['CHANNEL'] 이 비어 있습니다.")
    if s.startswith(("http://", "https://")):
        path = re.sub(r"^https?://(?:www\.|m\.)?youtube\.com", "", s)
        path = re.sub(r"/(?:%s)$" % "|".join(TABS_ALL), "", path)
        if not path.startswith("/"):
            raise ValueError(f"유튜브 채널 URL 이 아닙니다: {spec}")
        return "https://www.youtube.com" + path
    if s.startswith("@"):
        return f"https://www.youtube.com/{s}"
    if re.fullmatch(r"UC[0-9A-Za-z_-]{22}", s):
        return f"https://www.youtube.com/channel/{s}"
    return f"https://www.youtube.com/@{s}"      # @ 를 빠뜨린 핸들로 본다


def resolve_channel(spec):
    # 실제로 조회해 채널ID·채널명·핸들을 확정한다
    base = channel_base(spec)
    r = ytdlp("--flat-playlist", "-J", "--playlist-end", "1",
              "--extractor-args", "youtube:lang=ko", base + "/videos",
              timeout=CONFIG["TIMEOUT_INFO"])
    try:
        d = json.loads(r.stdout)
    except Exception:
        d = None
    if not isinstance(d, dict) or not d:
        tail = [l for l in (r.stderr or "").strip().split("\n")
                if l.strip() and not l.startswith("WARNING")]
        raise RuntimeError(
            f"채널을 찾지 못했습니다: {spec}\n  요청: {base}/videos\n  "
            + (tail[-1] if tail else "응답이 비어 있습니다")
            + "\n  → 핸들 철자를 확인하시거나 채널 ID(UC…)를 직접 넣어 보세요.")
    cid = d.get("channel_id") or d.get("uploader_id") or d.get("id") or ""
    if not re.fullmatch(r"UC[0-9A-Za-z_-]{22}", str(cid)):
        raise RuntimeError(f"채널 ID 를 확정하지 못했습니다: {spec} (얻은 값 {cid!r})")
    return {"channel_id": cid,
            "name": d.get("channel") or d.get("uploader") or "",
            "handle": d.get("uploader_id") or "",
            "url": f"https://www.youtube.com/channel/{cid}",
            "follower_count": d.get("channel_follower_count")}


CHANNEL_BASE = None     # 3단계에서 채워진다


def channel_url(tab):
    if CHANNEL_BASE is None:
        raise RuntimeError("3단계(채널 확인) 셀을 먼저 실행하세요.")
    return f"{CHANNEL_BASE}/{tab}"


print("유틸 준비 완료")

## 3. 채널 확인 · 폴더 준비

입력한 값을 실제로 조회해 **채널 ID·채널명·핸들**을 확정하고,
`generic` 프리셋이면 폴더명·파일명 규칙도 여기서 정합니다.

긴 수집을 시작하기 전에 **의도한 채널이 맞는지 여기서 확인**하세요.

In [ ]:
print(f"[3단계] 채널 확인 — 입력값: {CONFIG['CHANNEL']}")
CH = resolve_channel(CONFIG["CHANNEL"])
CHANNEL_BASE = CH["url"]

print(f"   채널명    : {CH['name']}")
print(f"   핸들      : {CH['handle']}")
print(f"   채널 ID   : {CH['channel_id']}")
if CH["follower_count"]:
    print(f"   구독자    : {CH['follower_count']:,}")

# ── 프리셋이 비워둔 이름들을 채널 정보로 채운다 ──────────────────────────────
_handle = (CH["handle"] or "").lstrip("@") or CH["channel_id"]
_slug   = re.sub(r"[^a-z0-9]+", "_", _handle.lower()).strip("_") or "channel"
CONFIG["SLUG"]         = CONFIG["SLUG"]         or _slug
CONFIG["MEDIA_DIR"]    = CONFIG["MEDIA_DIR"]    or _slug.upper()
CONFIG["MEDIA_NAME"]   = CONFIG["MEDIA_NAME"]   or (CH["name"] or _slug)
CONFIG["PROGRAM_NAME"] = CONFIG["PROGRAM_NAME"] or (CH["name"] or _slug)

ROOT  = CONFIG["OUT_ROOT"]
MEDIA = CONFIG["MEDIA_DIR"]
SLUG  = CONFIG["SLUG"]

RAW_DIR   = os.path.join(ROOT, "01_RAW", MEDIA)
RAW_SIS   = os.path.join(RAW_DIR, "sister")
RAW_XLSX  = os.path.join(RAW_DIR, "raw_xlsx")
TEXT_DIR  = os.path.join(ROOT, "03_TEXT", MEDIA)
TEXT_SIS  = os.path.join(TEXT_DIR, "sister")
META_DIR  = os.path.join(ROOT, "02_METADATA")
LOG_DIR   = os.path.join(ROOT, "04_LOG")

# 표에 적히는 상대경로 (프로젝트 루트 기준 — 폴더를 옮겨도 유효)
REL_TEXT = os.path.join("03_TEXT", MEDIA)
REL_SIS  = os.path.join(REL_TEXT, "sister")

for d in (RAW_DIR, RAW_SIS, RAW_XLSX,
          TEXT_DIR, TEXT_SIS, META_DIR, LOG_DIR):
    os.makedirs(d, exist_ok=True)

NAMES = {
    "meta_main":   f"{SLUG}_metadata.xlsx",
    "meta_sister": f"{SLUG}_sister_metadata.xlsx",
    "csv_broadcasts":        f"{SLUG}_broadcasts.csv",
    "csv_chapters":          f"{SLUG}_chapters.csv",
    "csv_sister_broadcasts": f"{SLUG}_sister_broadcasts.csv",
    "csv_sister_chapters":   f"{SLUG}_sister_chapters.csv",
    "log_main":    f"{SLUG}_collection_log.json",
    "log_sister":  f"{SLUG}_sister_collection_log.json",
    "raw_broadcast": f"RAW_{safe_filename(CONFIG['MEDIA_NAME'])}_방송단위",
    "raw_chapter":   f"RAW_{safe_filename(CONFIG['MEDIA_NAME'])}_챕터단위",
}
CONFIG["NAMES"] = NAMES


SEG_PREFIX = CONFIG["SEGMENT_PREFIX"] or re.sub(r"[^A-Z0-9]", "", SLUG.upper())[:3] or "SEG"

# 표의 열 순서 (원본 산출물과 동일하게 맞춘 기준)
BRO_COLS = ["date", "weekday", "video_id", "title", "url", "upload_date",
            "duration_sec", "duration_hms", "view_count_at_collect", "like_count",
            "comment_count", "auto_caption_ko", "manual_caption", "n_caption_lines",
            "n_chars", "n_chapters", "transcript_file", "flat_file", "raw_info",
            "raw_subtitle", "collected_at"]
SIS_BRO_COLS = ["date", "weekday", "program", "prog_code", "is_main_show", "video_id",
                "title", "url", "duration_sec", "duration_hms", "upload_date",
                "view_count_at_collect", "like_count", "comment_count", "auto_caption_ko",
                "n_caption_lines", "n_chars", "n_chapters", "transcript_file",
                "flat_file", "raw_info", "collected_at"]
CHAP_COLS = ["segment_id", "date", "video_id", "chapter_no", "chapter_title",
             "start", "end", "start_sec", "end_sec", "duration_min",
             "n_lines", "n_chars", "url_at", "transcript_file"]
SIS_CHAP_COLS = ["segment_id", "date", "program", "video_id", "chapter_no", "chapter_title",
                 "start", "end", "start_sec", "end_sec", "duration_min",
                 "n_lines", "n_chars", "url_at", "transcript_file"]


def text_stem(ymd, vid, code=None):
    # 자막 txt 의 파일명 조각
    if code:                                   # 파생 프로그램은 항상 코드+id
        return f"{ymd}_{code}_{vid}"
    return f"{ymd}_{SLUG}_{vid}" if CONFIG["STEM_WITH_VIDEO_ID"] else f"{ymd}_{SLUG}"


print(f"\n   출력 루트 : {ROOT}")
print(f"   미디어 폴더: 01_RAW/{MEDIA} · 03_TEXT/{MEDIA}")
print(f"   파일명 예  : {text_stem('20260723', 'aB3xYz')}_full.txt")
print("\n   ※ 의도한 채널이 아니면 CONFIG['CHANNEL'] 을 고치고 1단계부터 다시 실행하세요.")
print("   ※ 재현을 위해 위 채널 ID 를 기록해 두시길 권합니다(핸들은 바뀔 수 있습니다).")

## 4. 영상 열거

`--flat-playlist`로 채널의 각 탭을 훑습니다. 영상을 내려받지 않고 목록만 읽으므로 빠릅니다.
`--extractor-args youtube:lang=ko`는 제목이 자동 번역되는 것을 막습니다
(이게 없으면 제목 정규식이 영어로 번역된 제목에 걸리지 않습니다).

`DATE_SOURCE="upload_date"`면 목록만으로는 날짜를 알 수 없어 **영상마다 한 번 더 조회**합니다.
최신순으로 훑다가 기간보다 오래된 영상이 연속으로 나오면 멈추고, 조회 결과는 캐시에 남깁니다.

`generic` 프리셋에서는 모든 영상이 '본방송'으로 분류되고 파생 프로그램은 0편입니다.

In [ ]:
DATE_CACHE_PATH = os.path.join(LOG_DIR, "upload_date_cache.json")
DATE_CACHE = json.load(open(DATE_CACHE_PATH, encoding="utf-8")) \
             if os.path.exists(DATE_CACHE_PATH) else {}


def probe_upload_date(vid):
    # 업로드 날짜만 가볍게 확인하고 캐시한다
    if vid in DATE_CACHE:
        return DATE_CACHE[vid] or None
    r = ytdlp("--skip-download", "--no-warnings", "-O", "%(upload_date)s",
              f"https://www.youtube.com/watch?v={vid}", timeout=CONFIG["TIMEOUT_INFO"])
    d = (r.stdout or "").strip().split("\n")[0].strip()
    DATE_CACHE[vid] = d if re.fullmatch(r"\d{8}", d) else ""
    time.sleep(CONFIG["REQ_DELAY"])
    return DATE_CACHE[vid] or None


def scan_tab(tab, seen):
    # 탭 하나를 최신순으로 훑는다. seen 은 video_id → rec
    r = ytdlp("--flat-playlist", "-J", "--playlist-end", str(CONFIG["ENUM_LIMIT"]),
              "--extractor-args", "youtube:lang=ko", channel_url(tab),
              timeout=CONFIG["TIMEOUT_INFO"])
    try:
        entries = json.loads(r.stdout).get("entries") or []
    except Exception:
        tail = [l for l in (r.stderr or "").strip().split("\n")
                if l.strip() and not l.startswith("WARNING")]
        print(f"  ! /{tab} 조회 실패 — {tail[-1] if tail else '응답 없음'}")
        return 0, 0, {}

    by_date, n_new, old_streak, probed = CONFIG["PERIOD_MODE"] == "date", 0, 0, {}
    undated_here = {}
    for e in entries:
        vid, title = e.get("id"), str(e.get("title") or "")
        if not vid or vid in seen:
            continue

        ymd = None
        if by_date:
            if CONFIG["DATE_SOURCE"] == "title":
                ymd = date_from_title(title)
                if ymd is None:
                    undated_here[vid] = title
                    continue
            else:
                ymd = probe_upload_date(vid)
                probed[vid] = ymd
                if ymd is None:
                    undated_here[vid] = title
                    continue
            if ymd > CONFIG["END_DATE"]:
                continue                       # 기간보다 최신 — 계속 내려간다
            if ymd < CONFIG["START_DATE"]:
                old_streak += 1
                # 업로드일 기준일 때만 조기 종료 (목록이 최신순이므로 안전)
                if CONFIG["DATE_SOURCE"] == "upload_date" and old_streak >= CONFIG["PROBE_STOP_AFTER"]:
                    print(f"    · 기간 밖 {old_streak}편 연속 — /{tab} 탐색 중단")
                    break
                continue
            old_streak = 0
        else:
            if len(seen) >= CONFIG["MAX_VIDEOS"]:
                break

        rec = {"video_id": vid, "title": title, "date": ymd,
               "duration": e.get("duration"), "tab": tab}
        if is_main_show(title):
            if CONFIG["ONE_MAIN_PER_DAY"] and ymd and any(
                    v["date"] == ymd and v.get("_main") for v in seen.values()):
                continue                        # 같은 날 본방송은 먼저 본 것만
            rec["_main"] = True
        elif CONFIG["COLLECT_SISTER"]:
            prog = program_of(title)
            rec.update(_main=False, program=prog, code=code_of(prog))
        else:
            continue                            # 파생을 안 받으면 버린다
        seen[vid] = rec
        n_new += 1
    return len(entries), n_new, undated_here


print(f"[4단계] 영상 열거 — {CHANNEL_BASE}")
if CONFIG["PERIOD_MODE"] == "date" and CONFIG["DATE_SOURCE"] == "upload_date":
    print("   (업로드일을 영상마다 조회합니다. 처음 실행은 시간이 걸립니다)")

SEEN, UNDATED = {}, {}
for tab in CONFIG["TABS"]:
    n_all, n_new, und = scan_tab(tab, SEEN)
    UNDATED.update(und)
    n_m = sum(1 for v in SEEN.values() if v.get("_main"))
    print(f"  /{tab}: {n_all:4d}건 조회 → 신규 {n_new}건 "
          f"(본방송 {n_m} · 파생 {len(SEEN)-n_m})")

with open(DATE_CACHE_PATH, "w", encoding="utf-8") as f:
    json.dump(DATE_CACHE, f, ensure_ascii=False)

MAIN_VIDEOS   = {k: v for k, v in SEEN.items() if v.get("_main")}
SISTER_VIDEOS = {k: v for k, v in SEEN.items() if not v.get("_main")}
print(f"\n본방송 {len(MAIN_VIDEOS)}편 / 파생 {len(SISTER_VIDEOS)}편")
if UNDATED:
    print(f"날짜를 알 수 없어 제외된 영상 {len(UNDATED)}건 "
          f"(예: {list(UNDATED.values())[0][:36]}…)")

MISSING = []
if CONFIG["PERIOD_MODE"] == "date" and CONFIG["ONE_MAIN_PER_DAY"]:
    _have = {v["date"] for v in MAIN_VIDEOS.values()}
    print("\n기간 내 본방송 유무:")
    for ymd, wd in expected_days():
        if ymd in _have:
            print(f"  {ymd}({WEEKDAY[wd]}) ○")
        else:
            MISSING.append({"date": ymd, "weekday": WEEKDAY[wd],
                            "reason": "주말" if wd >= 5 else "미확인"})
            note = "주말(정규 방송 없음)" if wd >= 5 else "★ 평일인데 없음 — 확인 필요"
            print(f"  {ymd}({WEEKDAY[wd]}) ─ {note}")

if SISTER_VIDEOS:
    print("\n파생 프로그램:")
    for p, n in Counter(v["program"] for v in SISTER_VIDEOS.values()).most_common():
        mins = sum((v["duration"] or 0)//60 for v in SISTER_VIDEOS.values() if v["program"] == p)
        print(f"  {p:24s} {n:3d}편 {mins:6d}분")

if not SEEN:
    print("\n★ 영상이 하나도 잡히지 않았습니다. 아래를 확인하세요.")
    print("   · 제목에 날짜가 없는 채널이면 CONFIG['DATE_SOURCE'] = 'upload_date'")
    print("   · START_DATE / END_DATE 가 실제 업로드 기간과 겹치는지")
    print("   · ENUM_LIMIT 을 늘려 더 깊이 훑어야 할 수도 있습니다")

## 5. 메타데이터·자막 수집

영상 한 편당 두 가지를 받습니다.

| 파일 | 내용 |
|---|---|
| `{stem}.info.json` | yt-dlp 전체 메타데이터 (조회수·챕터·자막목록 등 원형) |
| `{stem}.ko-orig.json3` | 자동자막 원본, 밀리초 타임스탬프 보존 |

그리고 자막을 파싱해 `03_TEXT`에 두 판본을 씁니다.

* `_full.txt` — 챕터 구분선 + `HH:MM:SS 발화`. **인용한 발화를 영상에서 되찾을 수 있게** 하는 판본
* `_flat.txt` — 타임스탬프 없는 한 줄 평문. 검색·텍스트마이닝용

`01_RAW`의 원본은 이후 어떤 단계에서도 수정하지 않습니다.

### ⚠ `lang=ko` 를 영상 메타데이터 조회에 붙이면 안 됩니다

제목이 영어로 자동 번역되는 것을 막으려고 `--extractor-args youtube:lang=ko`를 쓰는데,
**이 옵션을 붙이면 유튜브가 일부 필드를 아예 내려주지 않습니다.**
같은 영상을 두 방식으로 받아 확인한 결과입니다.

| 필드 | `lang=ko` 포함 | `lang=ko` 없음 |
|---|---|---|
| `title` | 한국어 ○ | 한국어 ○ (원어가 한국어라 동일) |
| `view_count` | 23,204 | 23,204 |
| `comment_count` | 173 | 173 |
| **`upload_date`** | **None** | **20260823** |
| **`like_count`** | **None** | **3,321** |

그래서 이 노트북은 옵션을 **단계별로 나눠** 씁니다.

| 호출 | `lang=ko` | 이유 |
|---|---|---|
| 채널 목록 조회 | **붙임** | 제목 정규식이 번역된 제목에 걸리지 않게 |
| 영상 메타데이터 (`info.json`) | 붙이지 않음 | `upload_date` · `like_count` 보존 |
| 자막 | 붙임 | 자막 언어 선택에 필요 |

제목은 채널 목록에서 이미 한국어로 받아 두었으므로, 표에는 **목록 쪽 제목**을 씁니다.

> **조회수는 수집 시점 스냅샷**이라 열 이름에 `_at_collect`를 붙여 시점을 박아둡니다.
> `comment_count`는 유튜브가 반올림해 주는 경우가 있어(예: 정확히 `1000`) 참고값으로 보세요.

In [ ]:
def fetch_info(stem, video_id, raw_dir):
    # yt-dlp -J 전체 메타데이터. 있으면 재사용(재개 가능)
    # ※ 여기에는 lang=ko 를 넣지 않는다 — upload_date 와 like_count 가 통째로 사라진다.
    #   제목은 채널 목록에서 이미 한국어로 받아 두었으므로 그쪽을 쓴다.
    p = os.path.join(raw_dir, f"{stem}.info.json")
    if os.path.exists(p):
        return json.load(open(p, encoding="utf-8"))
    r = ytdlp("-J", "--no-warnings",
              f"https://www.youtube.com/watch?v={video_id}", timeout=CONFIG["TIMEOUT_INFO"])
    d = json.loads(r.stdout)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False)
    time.sleep(CONFIG["REQ_DELAY"])
    return d


def fetch_subs(stem, video_id, raw_dir):
    # 자동자막 원본을 01_RAW 에 저장. 이미 있으면 건너뛴다
    fmt = CONFIG["SUB_FORMAT"]
    target = os.path.join(raw_dir, f"{stem}.ko-orig.{fmt}")
    alt    = os.path.join(raw_dir, f"{stem}.ko.{fmt}")
    for p in (target, alt):
        if os.path.exists(p):
            return p
    args = ["--skip-download", "--write-auto-subs"]
    if CONFIG["WRITE_MANUAL_SUBS"]:
        args.append("--write-subs")
    ytdlp(*args, "--sub-langs", CONFIG["SUB_LANGS"], "--sub-format", fmt,
          "--extractor-args", "youtube:lang=ko",
          "-o", os.path.join(raw_dir, f"{stem}.%(ext)s"),
          f"https://www.youtube.com/watch?v={video_id}", timeout=CONFIG["TIMEOUT_SUB"])
    time.sleep(CONFIG["REQ_DELAY"])
    for p in (target, alt):
        if os.path.exists(p):
            return p
    return None


def write_transcripts(out_dir, stem_txt, header_lines, lines, chapters):
    # timestamp 보존본과 평문본을 함께 만든다
    def chapter_of(t):
        for c in chapters:
            if c["start_time"] <= t < c["end_time"]:
                return c["title"]
        return ""

    p1 = os.path.join(out_dir, f"{stem_txt}_full.txt")
    cur = None
    with open(p1, "w", encoding=ENC) as f:
        f.write("\n".join(header_lines))
        for t, txt in lines:
            ch = chapter_of(t)
            if ch != cur:
                cur = ch
                f.write(f"\n\n----- [챕터] {ch or '(구분 없음)'} -----\n")
            f.write(f"{hms(t)} {txt}\n")

    p2 = os.path.join(out_dir, f"{stem_txt}_flat.txt")
    with open(p2, "w", encoding=ENC) as f:
        f.write(" ".join(txt for _, txt in lines))
    return p1, p2


def transcript_header(ymd, info, n_lines, program, is_main=True, title=None):
    return [
        f"방송일      : {ymd_dash(ymd)} ({wd_of(ymd)})",
        f"프로그램    : {program}" + ("" if is_main else "   ※ 본방송 아님(파생 프로그램)"),
        f"영상 제목   : {title or info.get('title','')}",
        f"URL         : https://www.youtube.com/watch?v={info.get('id','')}",
        f"업로드 시각 : {info.get('upload_date','')} (timestamp={info.get('timestamp','')})",
        f"영상 길이   : {info.get('duration','')}초 ({hms(info.get('duration') or 0)})",
        f"조회수      : {info.get('view_count','')} (수집 시점 기준)",
        f"자막 종류   : 유튜브 자동생성(ko-orig)",
        f"자막 이벤트 : {n_lines}줄",
        f"수집 일시   : {now_str()}",
        "",
        "※ 이 파일은 영상 전체 자막이다. 관련 구간만 잘라내지 않았다.",
        "※ 자동생성 자막이므로 고유명사 오인식이 있다. 발화를 인용할 때는",
        "   반드시 timestamp 로 영상에 돌아가 확인할 것.",
        SEP, "",
    ]


def collect_videos(videos, raw_dir, text_dir, is_main, label):
    bro, chap = [], []
    items = sorted(videos.values(), key=lambda x: (x.get("date") or "", x["video_id"]))
    print(f"{label} ({len(items)}편)")
    for i, rec in enumerate(items, 1):
        vid  = rec["video_id"]
        code = None if is_main else rec.get("code")
        _d0 = rec.get("date")
        stem0 = ((f"{_d0}_{vid}" if _d0 else vid) if is_main
                 else (f"{_d0}_{code}_{vid}" if _d0 else f"{code}_{vid}"))
        try:
            info = fetch_info(stem0, vid, raw_dir)
        except Exception as e:
            print(f"   ! {stem0} 메타데이터 실패: {friendly_error(e)}")
            continue
        ymd = rec.get("date") or (info.get("upload_date") or "")
        # 파일명이 날짜에 의존하므로 확정된 날짜로 다시 맞춘다
        stem = f"{ymd}_{vid}" if is_main else f"{ymd}_{code}_{vid}"
        if stem != stem0:                       # 날짜를 알게 됐으니 파일명을 맞춘다
            for ext in (".info.json", f".ko-orig.{CONFIG['SUB_FORMAT']}",
                        f".ko.{CONFIG['SUB_FORMAT']}"):
                a, b = os.path.join(raw_dir, stem0+ext), os.path.join(raw_dir, stem+ext)
                if os.path.exists(a) and not os.path.exists(b):
                    os.replace(a, b)
        sub      = fetch_subs(stem, vid, raw_dir)
        lines    = parse_json3(sub) if sub else []
        chapters = info.get("chapters") or []
        auto     = info.get("automatic_captions") or {}
        prog     = CONFIG["PROGRAM_NAME"] if is_main else rec.get("program", "")
        tstem    = text_stem(ymd, vid, code)
        title    = rec.get("title") or info.get("title", "")   # 목록의 한국어 제목 우선

        if lines:
            write_transcripts(text_dir, tstem,
                              transcript_header(ymd, info, len(lines), prog, is_main, title),
                              lines, chapters)

        row = {
            "date": ymd_dash(ymd), "weekday": wd_of(ymd),
            "video_id": vid, "title": title,
            "url": f"https://www.youtube.com/watch?v={vid}",
            "upload_date": info.get("upload_date", ""),
            "duration_sec": info.get("duration", ""),
            "duration_hms": hms(info.get("duration") or 0),
            "view_count_at_collect": info.get("view_count", ""),
            "like_count": info.get("like_count", ""),
            "comment_count": info.get("comment_count", ""),
            "auto_caption_ko": 1 if ("ko-orig" in auto or "ko" in auto) else 0,
            "manual_caption": "|".join((info.get("subtitles") or {}).keys()),
            "n_caption_lines": len(lines),
            "n_chars": sum(len(t) for _, t in lines),
            "n_chapters": len(chapters),
            "transcript_file": f"{tstem}_full.txt",
            "flat_file": f"{tstem}_flat.txt",
            "raw_info": f"{stem}.info.json",
            "raw_subtitle": os.path.basename(sub) if sub else "",
            "collected_at": now_str(),
        }
        if not is_main:
            row = {"date": row["date"], "weekday": row["weekday"],
                   "program": prog, "prog_code": code, "is_main_show": 0,
                   **{k: v for k, v in row.items()
                      if k not in ("date", "weekday", "manual_caption", "raw_subtitle")}}
        bro.append(row)

        for j, c in enumerate(chapters, 1):
            st, en = c["start_time"], c["end_time"]
            seg = [t for t in lines if st <= t[0] < en]
            cr = {"segment_id": (f"{SEG_PREFIX}_{ymd}_{j:02d}" if is_main
                                 else f"{SEG_PREFIX}S_{ymd}_{code}_{j:02d}"),
                  "date": ymd_dash(ymd), "video_id": vid,
                  "chapter_no": j, "chapter_title": c["title"],
                  "start": hms(st), "end": hms(en),
                  "start_sec": int(st), "end_sec": int(en),
                  "duration_min": round((en - st) / 60, 1),
                  "n_lines": len(seg), "n_chars": sum(len(t) for _, t in seg),
                  "url_at": f"https://www.youtube.com/watch?v={vid}&t={int(st)}s",
                  "transcript_file": f"{tstem}_full.txt"}
            if not is_main:
                cr = {"segment_id": cr["segment_id"], "date": cr["date"], "program": prog,
                      **{k: v for k, v in cr.items() if k not in ("segment_id", "date")}}
            chap.append(cr)

        tag = f"{code:5s} " if code else ""
        print(f"   [{i:3d}/{len(items)}] {ymd_dash(ymd)}({wd_of(ymd)}) {tag}"
              f"자막 {len(lines):5d}줄 / {row['n_chars']:7,}자 / 챕터 {len(chapters):2d}개")
    return bro, chap

In [ ]:
BRO, CHAP = collect_videos(MAIN_VIDEOS, RAW_DIR, TEXT_DIR, True, "[5단계] 본방송 수집")

xl = os.path.join(META_DIR, NAMES["meta_main"])
with pd.ExcelWriter(xl, engine="openpyxl") as w:
    order_cols(pd.DataFrame(BRO), BRO_COLS).to_excel(w,  sheet_name="broadcasts", index=False)
    order_cols(pd.DataFrame(CHAP), CHAP_COLS).to_excel(w, sheet_name="chapters",   index=False)
    pd.DataFrame(MISSING).to_excel(w, sheet_name="missing_days", index=False)

log_p = os.path.join(LOG_DIR, NAMES["log_main"])
with open(log_p, "w", encoding="utf-8") as f:
    json.dump({"collected_at": now_str(), "preset": PRESET,
               "channel": CH, "period_mode": CONFIG["PERIOD_MODE"],
               "period": f"{CONFIG['START_DATE']}~{CONFIG['END_DATE']}",
               "n_broadcasts": len(BRO), "n_chapters": len(CHAP),
               "missing_days": MISSING,
               "total_chars": sum(b["n_chars"] for b in BRO),
               "note": "연구 목적 수집. 자동생성 자막이므로 고유명사 오인식 있음(검증 필요).",
               "broadcasts": BRO}, f, ensure_ascii=False, indent=1)

print(f"\n★ 본방송 {len(BRO)}편 / 챕터 {len(CHAP)}개 / 자막 {sum(b['n_chars'] for b in BRO):,}자")
print(f"   {os.path.relpath(xl, ROOT)}\n   {os.path.relpath(log_p, ROOT)}")

## 6. 파생(자매) 프로그램 수집

`COLLECT_SISTER = False`(`generic` 기본값)면 이 셀은 아무 일도 하지 않고 넘어갑니다.

본방송 판별 정규식(`MAIN_SHOW_PATTERN`)을 쓰는 프리셋에서, 본방송이 아닌 영상을
`sister/` 폴더와 `*_sister_*` 파일에 **완전히 분리**해 담는 단계입니다.
파일명은 `{YYYYMMDD}_{프로그램코드}_{video_id}` 형식입니다.


In [ ]:
SIS_BRO, SIS_CHAP = ([], [])
if SISTER_VIDEOS:
    SIS_BRO, SIS_CHAP = collect_videos(SISTER_VIDEOS, RAW_SIS, TEXT_SIS, False,
                                       "[6단계] 파생 프로그램 수집")
    xl2 = os.path.join(META_DIR, NAMES["meta_sister"])
    with pd.ExcelWriter(xl2, engine="openpyxl") as w:
        order_cols(pd.DataFrame(SIS_BRO), SIS_BRO_COLS).to_excel(w,  sheet_name="broadcasts", index=False)
        order_cols(pd.DataFrame(SIS_CHAP), SIS_CHAP_COLS).to_excel(w, sheet_name="chapters",   index=False)
    with open(os.path.join(LOG_DIR, NAMES["log_sister"]), "w", encoding="utf-8") as f:
        json.dump({"collected_at": now_str(),
                   "scope": "채널 파생 프로그램 (본방송 제외)",
                   "period": f"{CONFIG['START_DATE']}~{CONFIG['END_DATE']}",
                   "n_videos": len(SIS_BRO), "n_chapters": len(SIS_CHAP),
                   "total_chars": sum(r["n_chars"] for r in SIS_BRO),
                   "note": "분석 단위는 본방송이나 '빠짐없는 데이터' 원칙에 따라 별도 층으로 확보.",
                   "videos": SIS_BRO}, f, ensure_ascii=False, indent=1)
    print(f"\n★ 파생 {len(SIS_BRO)}편 / 챕터 {len(SIS_CHAP)}개 / "
          f"자막 {sum(r['n_chars'] for r in SIS_BRO):,}자")
else:
    print("[6단계] 파생 프로그램 — 수집 대상 없음 (건너뜀)")

## 7. ASR 고유명사 보정

`03_TEXT`의 원본 txt는 그대로 두고 `_corrected` 접미사로 별도 파일을 만듭니다.
보정 전 파일을 남기는 이유는 **보정이 옳았는지 대조할 수 있어야 하기 때문**입니다.

모든 치환은 `04_LOG/asr_correction_log.xlsx`에 위치·전후 문맥 40자와 함께 기록됩니다.

| 시트 | 내용 |
|---|---|
| `replacements` | 치환 한 건 한 건의 위치와 전후 문맥 |
| `dictionary` | 사전 전체 + 실제 적용 횟수 |
| `preserved` | 비슷하지만 치환하지 않은 것과 그 사유 |
| `recall_effect` | 키워드별 보정 전후 등장 횟수와 증가율 |

사전이 비어 있는 프리셋(`generic`)에서도 `_corrected` 파일은 **원본과 같은 내용으로** 만들어집니다.
뒤 단계의 경로 열이 항상 유효하도록 하기 위해서입니다.

In [ ]:
def correct_text(text, ordered, log_rows, source, ctx=40):
    # 문자열 치환 + 로그 기록. LLM 을 쓰지 않으므로 환각이 없다
    out = text
    for variant, repl, canon, conf, note in ordered:
        if variant not in out:
            continue
        # 치환하면 위치가 밀리므로 위치·문맥을 먼저 수집
        for m in re.finditer(re.escape(variant), out):
            log_rows.append({"source": source, "stage": "1_문자열", "variant": variant,
                             "replaced_with": repl, "canonical": canon,
                             "confidence": conf, "reason": note, "pos": m.start(),
                             "context_before": out[max(0, m.start()-ctx):m.start()],
                             "context_after": out[m.end():m.end()+ctx]})
        out = out.replace(variant, repl)

    for pat, repl, canon, conf, note in CONTEXT_CORRECTIONS:
        for m in re.finditer(pat, out):
            log_rows.append({"source": source, "stage": "2_문맥조건", "variant": m.group(0),
                             "replaced_with": repl, "canonical": canon,
                             "confidence": conf, "reason": f"{note}  [규칙 {pat}]", "pos": m.start(),
                             "context_before": out[max(0, m.start()-ctx):m.start()],
                             "context_after": out[m.end():m.end()+ctx]})
        out = re.sub(pat, repl, out)
    return out


# 긴 변이형부터 치환 (부분 문자열 오작동 방지)
ordered = sorted(CORRECTIONS, key=lambda x: -len(x[0]))

files = [f for f in (sorted(glob.glob(os.path.join(TEXT_DIR, "*_full.txt"))) +
                     sorted(glob.glob(os.path.join(TEXT_DIR, "*_flat.txt"))) +
                     sorted(glob.glob(os.path.join(TEXT_SIS, "*_full.txt"))) +
                     sorted(glob.glob(os.path.join(TEXT_SIS, "*_flat.txt"))))
         if "_corrected" not in f]

print(f"[7단계] 고유명사 보정 ({len(files)}개 파일 · 사전 {len(CORRECTIONS)}건)")
log_rows, before_all, after_all = [], "", ""
for fp in files:
    src = open(fp, encoding=ENC).read()
    dst = correct_text(src, ordered, log_rows, os.path.basename(fp)) if ordered or CONTEXT_CORRECTIONS else src
    with open(fp.replace(".txt", "_corrected.txt"), "w", encoding=ENC) as f:
        f.write(dst)
    if fp.endswith("_flat.txt"):        # 통계는 평문본 기준(중복 방지)
        before_all += " " + src
        after_all  += " " + dst

df_log = pd.DataFrame(log_rows)
b = {t: before_all.count(t) for t in RECALL_TERMS}
a = {t: after_all.count(t)  for t in RECALL_TERMS}

xl = os.path.join(LOG_DIR, "asr_correction_log.xlsx")
with pd.ExcelWriter(xl, engine="openpyxl") as w:
    (df_log if len(df_log) else pd.DataFrame(columns=["source"])) \
        .to_excel(w, sheet_name="replacements", index=False)
    pd.DataFrame(
        [{"변이형": v, "치환형": r, "표준형": c, "confidence": cf, "근거": n,
          "적용횟수": int((df_log["variant"] == v).sum()) if len(df_log) else 0}
         for v, r, c, cf, n in CORRECTIONS]
        + [{"변이형": f"[정규식] {v}", "치환형": r, "표준형": c, "confidence": cf, "근거": n,
            "적용횟수": int(df_log["reason"].astype(str).str.contains(re.escape(v), regex=True).sum())
                        if len(df_log) else 0}
           for v, r, c, cf, n in CONTEXT_CORRECTIONS]
        or [{"변이형": "(사전 없음)"}]
    ).to_excel(w, sheet_name="dictionary", index=False)
    pd.DataFrame([{"변이형": v, "검토대상": t, "보존사유": n} for v, t, n in PRESERVED]
                 or [{"변이형": "(없음)"}]).to_excel(w, sheet_name="preserved", index=False)
    pd.DataFrame([{"용어": k, "보정전": b[k], "보정후": a[k], "증가": a[k]-b[k],
                   "증가율(%)": round((a[k]-b[k])/b[k]*100, 1) if b[k] else None}
                  for k in b] or [{"용어": "(키워드 없음)"}]
                 ).to_excel(w, sheet_name="recall_effect", index=False)

with open(os.path.join(LOG_DIR, "asr_correction_dict.json"), "w", encoding="utf-8") as f:
    json.dump({"built_at": now_str(), "preset": PRESET,
               "method": "자모 편집거리 발굴 → LLM 문맥 판정 → 결정론적 치환",
               "corrections": [dict(zip(["variant","replace_with","canonical","confidence","note"], c))
                               for c in CORRECTIONS],
               "preserved": [dict(zip(["variant","checked_against","reason"], p)) for p in PRESERVED]},
              f, ensure_ascii=False, indent=1)

print(f"총 치환 {len(df_log)}건 → 04_LOG/asr_correction_log.xlsx")
if RECALL_TERMS:
    print(f"\n{'용어':10s} {'보정전':>7s} {'보정후':>7s} {'증가율':>8s}")
    for k in b:
        rate = f"{(a[k]-b[k])/b[k]*100:+.1f}%" if b[k] else "─"
        print(f"  {k:10s} {b[k]:7d} {a[k]:7d} {rate:>8s}")

## 8~9. 표 생성

### 왜 자막 전문을 표에 넣지 않는가

Excel 한 셀의 상한은 **32,767자**입니다. 긴 영상 한 편의 자막은 이 상한을 쉽게 넘습니다.
한계를 넘기면 Excel은 그냥 자르는 게 아니라 **행 구조 자체를 깨뜨려** 나머지를 옆 칸·아랫줄로
흘려보냅니다. 열과 내용이 전부 어긋나 보입니다. 게다가 **예외 없이 조용히** 잘립니다.

그래서 표에는 `text_len`(총 길이)과 **경로 네 개**만 넣고, 전문은 `03_TEXT`의 txt에 둡니다.

| 열 | 내용 |
|---|---|
| `txt_flat` | 평문(줄바꿈 없음), 보정 전 |
| `txt_full` | 타임스탬프본, 보정 전 |
| `txt_flat_corrected` | 평문, 보정 후 |
| `txt_full_corrected` | 타임스탬프본, 보정 후 — **분석에는 보통 이것** |

경로는 **프로젝트 루트 기준 상대경로**입니다. 폴더를 옮기거나 이름을 바꿔도 유효합니다.

```python
import pandas as pd, os
ROOT = r"...\youtube_woni"
df  = pd.read_csv(os.path.join(ROOT, "02_METADATA", "woni_broadcasts.csv"),
                  encoding="utf-8-sig")
txt = open(os.path.join(ROOT, df.loc[0, "txt_full_corrected"]), encoding="utf-8-sig").read()
```

전문을 표 안에 직접 넣고 싶으면 `CONFIG["TABLE_INCLUDE_TEXT"] = True`로 두세요.
CSV(UTF-8 BOM, 전 필드 따옴표)는 무손실이지만, XLSX는 32,000자 단위로 `text` / `text_2` / `text_3`에
나눠 담고 이어 붙여야 원문이 복원됩니다.

> **경고** — CSV를 Excel로 **열어서 다시 저장**하면 그 시점에 32,767자로 잘립니다.
> pandas·R(readr)로 읽을 때만 안전합니다.


### 두 층의 차이

| 위치 | 성격 |
|---|---|
| `01_RAW/.../raw_xlsx/` | **판정 이전 날것.** 보정하지 않은 원본 자막 기준. 영상 전체 한 표 |
| `02_METADATA/` | **보정본 기준.** 본방송/파생 분리, xlsx + CSV 정본 |

In [ ]:
def flatten(s):
    return re.sub(r"\s+", " ", str(s or "")).strip()


def read_txt(path):
    return open(path, encoding=ENC).read().strip() if os.path.exists(path) else ""


def lines_of(path):
    # 타임스탬프 파일 → [(초, 발화)]
    out = []
    if not os.path.exists(path):
        return out
    for ln in open(path, encoding=ENC):
        m = re.match(r"^(\d{2}):(\d{2}):(\d{2})\s(.*)$", ln.rstrip("\n"))
        if m:
            out.append((int(m.group(1))*3600 + int(m.group(2))*60 + int(m.group(3)), m.group(4)))
    return out


def text_cells(s, prefix="text"):
    # TABLE_INCLUDE_TEXT 가 False 면 길이만 남긴다
    s = str(s or "")
    if not CONFIG["TABLE_INCLUDE_TEXT"]:
        return {f"{prefix}_len": len(s)}
    n = CONFIG["CELL_MAX"]
    out = {prefix: s[:n]}
    if len(s) > n:
        out[f"{prefix}_2"] = s[n:2*n]
    if len(s) > 2*n:
        out[f"{prefix}_3"] = s[2*n:3*n]
    if len(s) > 3*n:
        out[f"{prefix}_3"] += "…[초과분은 03_TEXT의 txt 참조]"
    out[f"{prefix}_len"] = len(s)
    return out


def txt_paths(transcript_file, is_main):
    # transcript_file(=..._full.txt) → 네 판본의 루트 기준 상대경로
    base = REL_TEXT if is_main else REL_SIS
    stem = str(transcript_file).replace("_full.txt", "")
    return {"txt_flat":           os.path.join(base, f"{stem}_flat.txt"),
            "txt_full":           os.path.join(base, f"{stem}_full.txt"),
            "txt_flat_corrected": os.path.join(base, f"{stem}_flat_corrected.txt"),
            "txt_full_corrected": os.path.join(base, f"{stem}_full_corrected.txt")}


def save_table(df, csv_path, xlsx_path=None):
    df.to_csv(csv_path, index=False, encoding=ENC, quoting=csv.QUOTE_ALL)
    if xlsx_path and len(df) <= CONFIG["XLSX_ROW_LIMIT"]:
        try:
            df.to_excel(xlsx_path, index=False)
        except PermissionError:
            print(f"    ⚠ {os.path.basename(xlsx_path)} 이 열려 있어 건너뜀. CSV 정본은 저장됨.")


def verify_paths(df):
    miss = 0
    for c in ("txt_flat", "txt_full", "txt_flat_corrected", "txt_full_corrected"):
        if c in df.columns:
            miss += sum(1 for v in df[c].dropna()
                        if v and not os.path.exists(os.path.join(ROOT, v)))
    return miss


TXT_COLS = ["text", "text_2", "text_3", "text_len",
            "txt_flat", "txt_full", "txt_flat_corrected", "txt_full_corrected"]

In [ ]:
print("[8단계] 01_RAW 날것 표 (보정 전 자막 기준)")

rows = []
for src, is_main, tdir in ((BRO, 1, TEXT_DIR), (SIS_BRO, 0, TEXT_SIS)):
    for r in src:
        raw = read_txt(os.path.join(tdir, r["flat_file"]))
        rows.append({"media": CONFIG["MEDIA_NAME"],
                     "program": CONFIG["PROGRAM_NAME"] if is_main else r.get("program", ""),
                     "is_main_show": is_main, "date": r["date"], "weekday": r["weekday"],
                     "title": r["title"], "url": r["url"], "video_id": r["video_id"],
                     "upload_date": r["upload_date"], "duration_hms": r["duration_hms"],
                     "duration_sec": r["duration_sec"],
                     "view_count_at_collect": r["view_count_at_collect"],
                     "n_caption_lines": r["n_caption_lines"], "n_chapters": r["n_chapters"],
                     **text_cells(flatten(raw)),
                     "transcript_file": r["transcript_file"],
                     **txt_paths(r["transcript_file"], bool(is_main))})

if rows:
    df_b = pd.DataFrame(rows).sort_values(["date", "is_main_show", "program"],
                                          ascending=[True, False, True]).reset_index(drop=True)
    df_b = order_cols(df_b, ["media","program","is_main_show","date","weekday","title","url",
                             "video_id","upload_date","duration_hms","duration_sec",
                             "view_count_at_collect","n_caption_lines","n_chapters",
                             "text","text_2","text_3","text_len","transcript_file",
                             "txt_flat","txt_full","txt_flat_corrected","txt_full_corrected"])
    p = os.path.join(RAW_XLSX, NAMES["raw_broadcast"])
    save_table(df_b, p + ".csv", p + ".xlsx")
    print(f"  {os.path.basename(p)}  {len(df_b)}행 "
          f"(본방송 {int(df_b.is_main_show.sum())} + 파생 {int((1-df_b.is_main_show).sum())}) "
          f"· 경로 누락 {verify_paths(df_b)}건")

crows, cache = [], {}
for src, is_main, tdir in ((CHAP, 1, TEXT_DIR), (SIS_CHAP, 0, TEXT_SIS)):
    for c in src:
        f = c["transcript_file"]
        key = (tdir, f)
        if key not in cache:
            cache[key] = lines_of(os.path.join(tdir, f))     # 보정 전
        seg = [x for t, x in cache[key] if c["start_sec"] <= t < c["end_sec"]]
        crows.append({"media": CONFIG["MEDIA_NAME"],
                      "program": CONFIG["PROGRAM_NAME"] if is_main else c.get("program", ""),
                      "is_main_show": is_main,
                      "segment_id": c["segment_id"], "date": c["date"],
                      "chapter_no": c["chapter_no"], "chapter_title": c["chapter_title"],
                      "start": c["start"], "end": c["end"],
                      "start_sec": c["start_sec"], "end_sec": c["end_sec"],
                      "duration_min": c["duration_min"], "n_lines": c["n_lines"],
                      **text_cells(flatten(" ".join(seg))),
                      "url": c["url_at"], "video_id": c["video_id"],
                      "transcript_file": f,
                      **txt_paths(f, bool(is_main))})

if crows:
    df_c = pd.DataFrame(crows).sort_values(["date", "is_main_show", "start_sec"],
                                           ascending=[True, False, True]).reset_index(drop=True)
    df_c = order_cols(df_c, ["media","program","is_main_show","segment_id","date","chapter_no",
                             "chapter_title","start","end","start_sec","end_sec","duration_min",
                             "n_lines","text","text_2","text_3","text_len","url","video_id",
                             "transcript_file",
                             "txt_flat","txt_full","txt_flat_corrected","txt_full_corrected"])
    p = os.path.join(RAW_XLSX, NAMES["raw_chapter"])
    save_table(df_c, p + ".csv", p + ".xlsx")
    print(f"  {os.path.basename(p)}  {len(df_c)}행 · 경로 누락 {verify_paths(df_c)}건")
else:
    print("  (챕터가 있는 영상이 없습니다 — 챕터 표 생략)")

In [ ]:
print("[9단계] 02_METADATA 표 (보정본 자막 기준)")

def corrected(name):
    return str(name).replace(".txt", "_corrected.txt")


def build_broadcast_table(bro, tdir, is_main):
    return pd.DataFrame([{**r, **text_cells(flatten(read_txt(os.path.join(tdir, corrected(r["flat_file"])))))
                          , **txt_paths(r["transcript_file"], is_main)} for r in bro])


def build_chapter_table(chap, tdir, is_main):
    out, cache = [], {}
    for c in chap:
        f = corrected(c["transcript_file"])
        if f not in cache:
            cache[f] = lines_of(os.path.join(tdir, f))
        seg = [x for t, x in cache[f] if c["start_sec"] <= t < c["end_sec"]]
        out.append({**c, **text_cells(flatten(" ".join(seg))),
                    **txt_paths(c["transcript_file"], is_main)})
    return pd.DataFrame(out)


m_bro  = order_cols(build_broadcast_table(BRO,  TEXT_DIR, True),  BRO_COLS + TXT_COLS)
m_chap = order_cols(build_chapter_table(CHAP,   TEXT_DIR, True),  CHAP_COLS + TXT_COLS)
save_table(m_bro,  os.path.join(META_DIR, NAMES["csv_broadcasts"]),
           os.path.join(META_DIR, NAMES["csv_broadcasts"].replace(".csv", ".xlsx")))
save_table(m_chap, os.path.join(META_DIR, NAMES["csv_chapters"]),
           os.path.join(META_DIR, NAMES["csv_chapters"].replace(".csv", ".xlsx")))
with pd.ExcelWriter(os.path.join(META_DIR, NAMES["meta_main"]), engine="openpyxl") as w:
    m_bro.to_excel(w,  sheet_name="broadcasts",   index=False)
    m_chap.to_excel(w, sheet_name="chapters",     index=False)
    pd.DataFrame(MISSING).to_excel(w, sheet_name="missing_days", index=False)
print(f"  {NAMES['csv_broadcasts']:40s} {len(m_bro):5d}행 · 경로 누락 {verify_paths(m_bro)}건")
print(f"  {NAMES['csv_chapters']:40s} {len(m_chap):5d}행 · 경로 누락 {verify_paths(m_chap)}건")

s_bro = s_chap = pd.DataFrame()
if SIS_BRO:
    s_bro  = order_cols(build_broadcast_table(SIS_BRO, TEXT_SIS, False), SIS_BRO_COLS + TXT_COLS)
    s_chap = order_cols(build_chapter_table(SIS_CHAP,  TEXT_SIS, False), SIS_CHAP_COLS + TXT_COLS)
    save_table(s_bro,  os.path.join(META_DIR, NAMES["csv_sister_broadcasts"]),
               os.path.join(META_DIR, NAMES["csv_sister_broadcasts"].replace(".csv", ".xlsx")))
    save_table(s_chap, os.path.join(META_DIR, NAMES["csv_sister_chapters"]),
               os.path.join(META_DIR, NAMES["csv_sister_chapters"].replace(".csv", ".xlsx")))
    with pd.ExcelWriter(os.path.join(META_DIR, NAMES["meta_sister"]), engine="openpyxl") as w:
        s_bro.to_excel(w,  sheet_name="broadcasts", index=False)
        s_chap.to_excel(w, sheet_name="chapters",   index=False)
    print(f"  {NAMES['csv_sister_broadcasts']:40s} {len(s_bro):5d}행 · 경로 누락 {verify_paths(s_bro)}건")
    print(f"  {NAMES['csv_sister_chapters']:40s} {len(s_chap):5d}행 · 경로 누락 {verify_paths(s_chap)}건")

print("\n※ 자막 전문은 표에 없습니다. txt_full_corrected 열의 경로를 여세요.")

## 10. 검증

수집 결과의 자체 정합성을 점검합니다.
`CONFIG["REFERENCE_ROOT"]`에 비교할 기존 폴더 경로를 넣어두면 파일명·열 구성까지 대조합니다.

대조 항목은 넷입니다.

1. **파일 개수** — 원자료·자막 txt 4판본이 편수 × 판본수와 맞는지
2. **파일명 집합** — 기준 폴더와 정확히 같은 이름이 생겼는지 (누락/초과를 양쪽으로)
3. **표의 행 수·열 구성**
4. **자막 분량** — 편별 글자 수 (조회수는 시점 스냅샷이라 반드시 달라집니다)

In [ ]:
REF = CONFIG["REFERENCE_ROOT"]
has_ref = bool(REF) and os.path.isdir(REF)
print(f"[10단계] 검증   기준 폴더: {REF if has_ref else '(없음 — 자체 점검만)'}\n")

def names_in(d, pat="*"):
    return {os.path.basename(p) for p in glob.glob(os.path.join(d, pat))} if os.path.isdir(d) else set()

n_main, n_sis = len(BRO), len(SIS_BRO)
print("1) 파일 개수")
for label, got, exp in [
    ("01_RAW 본방송 info.json",   len(names_in(RAW_DIR, "*.info.json")), n_main),
    ("01_RAW 본방송 자막(ko-orig)", len(names_in(RAW_DIR, f"*.ko-orig.{CONFIG['SUB_FORMAT']}")),
                                  sum(1 for b in BRO if b["raw_subtitle"])),
    ("03_TEXT 본방송 txt(4판본)", len(names_in(TEXT_DIR, "*.txt")),
                                  sum(1 for b in BRO if b["n_caption_lines"])*4),
    ("01_RAW 파생 info.json",     len(names_in(RAW_SIS, "*.info.json")), n_sis),
    ("03_TEXT 파생 txt(4판본)",   len(names_in(TEXT_SIS, "*.txt")),
                                  sum(1 for b in SIS_BRO if b["n_caption_lines"])*4),
]:
    print(f"   {'OK ' if got == exp else '!! '} {label:28s} {got:5d} / 기대 {exp:5d}")

print("\n2) 기준 폴더와 파일명 대조")
if has_ref:
    for label, new_d, ref_d, pat in [
        (f"01_RAW/{MEDIA}",  RAW_DIR,  os.path.join(REF, "01_RAW", MEDIA),           "*.json"),
        (f"03_TEXT/{MEDIA}", TEXT_DIR, os.path.join(REF, "03_TEXT", MEDIA),          "*.txt"),
        ("01_RAW/.../sister",  RAW_SIS,  os.path.join(REF, "01_RAW", MEDIA, "sister"),  "*.json"),
        ("03_TEXT/.../sister", TEXT_SIS, os.path.join(REF, "03_TEXT", MEDIA, "sister"), "*.txt"),
    ]:
        new, ref = names_in(new_d, pat), names_in(ref_d, pat)
        if not ref:
            continue
        only_ref, only_new = sorted(ref - new), sorted(new - ref)
        flag = "OK " if not only_ref and not only_new else "!! "
        print(f"   {flag}{label:22s} 일치 {len(new & ref):4d} · 기준에만 {len(only_ref):3d} "
              f"· 새것에만 {len(only_new):3d}")
        for x in only_ref[:3]:
            print(f"        기준에만: {x}")
        for x in only_new[:3]:
            print(f"        새것에만: {x}")
else:
    print("   (기준 폴더 없음 — 건너뜀)")

print("\n3) 표 대조")
pairs = [(NAMES["csv_broadcasts"], m_bro), (NAMES["csv_chapters"], m_chap)]
if len(s_bro):
    pairs += [(NAMES["csv_sister_broadcasts"], s_bro), (NAMES["csv_sister_chapters"], s_chap)]
for name, df in pairs:
    line = f"   {name:40s} {len(df):7,}행 {len(df.columns):3d}열"
    rp = os.path.join(REF, "02_METADATA", name)
    if has_ref and os.path.exists(rp):
        rd = pd.read_csv(rp, dtype=str, nrows=0, encoding=ENC)
        miss  = [c for c in rd.columns if c not in df.columns]
        extra = [c for c in df.columns if c not in rd.columns]
        print(line + f"  | 기준 {len(rd.columns)}열 · 없는열 {len(miss)} · 추가열 {len(extra)}")
        if miss:
            print(f"        기준에만 있는 열: {', '.join(miss)}")
        if extra:
            print(f"        새것에만 있는 열: {', '.join(extra)}")
    else:
        print(line + ("  | (기준 없음)" if has_ref else ""))

print("\n4) 편별 자막 글자 수 (조회수는 시점 스냅샷이라 달라지는 게 정상)")
rp = os.path.join(REF, "02_METADATA", NAMES["csv_broadcasts"])
if has_ref and os.path.exists(rp):
    ref_df = pd.read_csv(rp, dtype=str, encoding=ENC)
    ref_map = dict(zip(ref_df["date"], ref_df["n_chars"].astype(int)))
    print(f"   {'날짜':12s} {'새 수집':>9s} {'기준':>9s} {'차이':>9s}")
    for _, r in m_bro.iterrows():
        o = ref_map.get(r["date"])
        d = "─" if o is None else f"{int(r['n_chars']) - o:+,}"
        print(f"   {r['date']:12s} {int(r['n_chars']):9,} {(o or 0):9,} {d:>9s}")
else:
    for _, r in m_bro.head(20).iterrows():
        print(f"   {r['date']:12s} {int(r['n_chars']):9,}자 · 챕터 {int(r['n_chapters']):2d}개")
    if len(m_bro) > 20:
        print(f"   … 외 {len(m_bro)-20}편")

print("\n" + SEP)
print(f"완료 — {ROOT}")

---

## 다른 채널 수집하기

1단계에서 두 줄만 고치면 됩니다.

```python
PRESET = "generic"
CONFIG["CHANNEL"] = "@수집할채널핸들"
```

기간을 쓰려면 `PERIOD_MODE="date"`와 `START_DATE`/`END_DATE`를,
최신 것만 보려면 `PERIOD_MODE="latest"`와 `MAX_VIDEOS`를 맞추세요.
`OUT_ROOT`는 채널마다 다르게 두시는 편이 안전합니다.

폴더명·파일명은 채널 핸들에서 자동으로 정해지므로 손댈 것이 없습니다.

## 이 노트북에 없는 것

| 항목 | 왜 |
|---|---|
| 주제 관련 구간 LLM 판정 | API 키가 필요하고 판정 기준이 연구 주제마다 다릅니다. 대신 `*_chapters.csv`(챕터 전수)가 나옵니다 |
| 영상 파일 다운로드 | 자막·메타데이터만 받습니다 |
| 댓글 | 자막·메타데이터만 받습니다 (영상별 댓글 수는 `comment_count` 열에 있음) |

## 자주 나는 문제

| 증상 | 원인·해결 |
|---|---|
| `채널을 찾지 못했습니다` | 핸들 철자 오류. 주소창의 `@...`를 그대로 복사하거나 채널 ID(`UC…`)를 넣으세요 |
| 채널 목록이 0건 | `yt-dlp`가 오래됨 → 0단계 셀 다시 실행 |
| `/streams: 조회 실패` | 그 채널에 라이브 탭이 없음. `CONFIG["TABS"] = ("videos",)` |
| 영상이 하나도 안 잡힘 | `PRESET="generic"`으로 두고 `DATE_SOURCE="upload_date"` 확인 |
| 자막이 전부 비어 있음 | 자동자막이 아직 생성되지 않은 영상. 며칠 뒤 재실행하면 채워집니다 |
| `like_count`·`upload_date`가 전부 빔 | `lang=ko`가 섞인 것. 영상 메타데이터 호출에는 넣지 않아야 합니다 |
| `Sign in to confirm you're not a bot` | `CONFIG["COOKIES_FROM_BROWSER"] = "chrome"` |
| 특정 영상만 계속 실패 | 그 `.info.json` / `.json3`을 지우고 재실행 (나머지는 건너뜁니다) |
| xlsx 저장이 안 됨 | 그 파일을 Excel로 열어둔 상태입니다. 닫고 재실행. CSV 정본은 이미 저장돼 있습니다 |
| 한글이 깨짐 | CSV는 UTF-8 BOM입니다. `pd.read_csv(..., encoding="utf-8-sig")` |

## 재실행에 대하여

4~6단계는 이미 받은 파일이 있으면 건너뜁니다. 중간에 멈춰도 그냥 다시 실행하세요.
**다시 받고 싶으면** 해당 `.info.json` · `.json3`을 지우면 됩니다.
7~9단계는 매번 다시 계산하므로 여러 번 돌려도 안전합니다.

## 수집 윤리

공개된 자료를 연구 목적으로 모으는 것이지만, 자막에는 출연자의 발화가 그대로 담깁니다.
인용·공개 단계에서는 연구 목적에 필요한 범위로 한정해 주세요.
요청 간 대기(`REQ_DELAY`)를 지나치게 줄이지 마세요 — 서버에 부담을 주고 차단됩니다.